# Z2005 Week 09 — Greedy Algorithms

A self-study notebook on the greedy paradigm: when local optimal choices lead to a global optimum, and when they do not.


## Learning Objectives

By the end of this notebook you will be able to:

- State the two properties (greedy-choice property and optimal substructure) that a problem must have for a greedy algorithm to be provably correct.
- Solve the activity-selection / interval-scheduling problem with a greedy algorithm and explain why "earliest finish time first" beats other plausible-sounding orderings.
- Distinguish fractional knapsack (greedy-solvable) from 0/1 knapsack (needs dynamic programming) and explain *why* the greedy strategy breaks for the 0/1 version.
- Build a Huffman code for a symbol-frequency table and use it to encode and decode a message.
- Explain why the coin-change greedy strategy is only correct for canonical coin systems, with a concrete counterexample.
- Given a new optimization problem, decide whether to reach for greedy or dynamic programming first, and justify the choice.


## How to use this notebook

Run the cells top to bottom. Cells marked `# TODO` are for you to complete — replace the
`raise NotImplementedError` with working code. Cells under **Self-Check** headings use
`assert` statements: they raise an `AssertionError` (or print nothing useful) if your code
is wrong, and print a friendly `✅` message if it is correct. The **Solutions** section at
the end has fully worked answers — try the exercises yourself first.


## 1. The greedy paradigm: what it is, and when it works

A **greedy algorithm** builds a solution one step at a time, and at each step it makes the
choice that looks best *right now*, without reconsidering it later. There is no backtracking
and no look-ahead. This makes greedy algorithms fast — typically a single pass over sorted
input — but it also means they only produce the *globally* optimal answer for a specific
class of problems.

A problem is a good candidate for a correct greedy algorithm when it has two properties:

1. **Greedy-choice property.** A globally optimal solution can be reached by making a
   locally optimal (greedy) choice first, then solving the remaining subproblem. In other
   words, you never have to revisit an early choice once a later part of the problem is
   examined.
2. **Optimal substructure.** An optimal solution to the problem contains optimal solutions
   to its subproblems. (This property is shared with dynamic programming — see the
   comparison at the end of this section.)

The common pitfall is *assuming* the greedy-choice property holds because a greedy strategy
"sounds reasonable." It has to be **proved**, usually with an *exchange argument*: take any
optimal solution that disagrees with the greedy choice, and show it can be transformed into
one that agrees with the greedy choice without making it worse. We will do this concretely
for activity selection below, and use brute-force search to show what happens when a
plausible-sounding greedy strategy is actually wrong.

**Greedy vs. dynamic programming, briefly:** both techniques rely on optimal substructure,
but DP additionally requires solving overlapping subproblems and combining them (usually
with memoization or a table), because the locally best choice is *not* always safe. If a
single sorted pass with no revisiting is provably correct, use greedy — it is simpler and
faster. If the locally best choice can trap you into a worse global outcome (verified by a
counterexample, not just intuition), fall back to DP. Week 08 covered DP in depth; this
notebook revisits coin change and knapsack specifically to contrast the two approaches on
the same problems.


## 2. Activity selection (interval scheduling)

**Problem:** given a set of activities, each with a start and finish time, and a single
resource (a meeting room, a machine, a lecture hall), select the maximum number of
non-overlapping activities.

The greedy strategy that works is: **sort activities by finish time, and repeatedly pick
the next activity whose start time is not before the finish time of the last one picked.**
Sorting by finish time (not by start time, not by duration, not by the number of
conflicts) is the crucial insight — finishing early leaves the most room for future
choices, which is exactly the intuition the exchange argument below makes rigorous.


In [ ]:
from typing import List, Tuple

def activity_selection(activities: List[Tuple[int, int]]) -> List[Tuple[int, int]]:
    """Select the maximum-size set of mutually compatible activities.

    Args:
        activities: list of (start, finish) pairs.

    Returns:
        A list of the selected (start, finish) pairs, in the order chosen.
    """
    # Sort by finish time -- this is the greedy choice that is provably optimal.
    ordered = sorted(activities, key=lambda a: a[1])
    selected = [ordered[0]]          # always take the first-finishing activity
    last_finish = ordered[0][1]
    for start, finish in ordered[1:]:
        if start >= last_finish:     # compatible: does not overlap the last pick
            selected.append((start, finish))
            last_finish = finish     # update the bar the next activity must clear
    return selected


meetings = [(1, 4), (3, 5), (0, 6), (5, 7), (3, 9), (5, 9), (6, 10), (8, 11), (8, 12), (2, 14), (12, 16)]
chosen = activity_selection(meetings)
print("selected:", chosen)
print("count:", len(chosen))  # classic textbook instance: optimal count is 4


### 2.1 Why "earliest finish time first" is correct: the exchange argument

Suppose, for contradiction, that some optimal solution `O` does not start with the
activity of earliest finish time, call it `a1`. Let `o1` be the first activity in `O`
(ordered by finish time). Since `a1` finishes no later than `o1`, we can swap `o1` out for
`a1` without breaking compatibility with the rest of `O` — everything after `o1` already
started at or after `o1`'s finish time, and `a1` finishes even earlier, so it is still
compatible. The swapped solution has the same size as `O`. Applying this argument
inductively along the rest of the schedule shows the fully greedy solution is optimal.

The cell below verifies this empirically: it checks the greedy answer against **brute-force
search over all subsets**, and separately shows that a superficially reasonable alternative
strategy — "shortest duration first" — is *not* always optimal.


In [ ]:
from itertools import combinations

def brute_force_max_compatible(activities: List[Tuple[int, int]]) -> int:
    """Try every subset and return the size of the largest compatible one.

    Only feasible for small inputs -- exponential time -- used here purely to validate
    the greedy algorithm against ground truth.
    """
    n = len(activities)
    best = 0
    for size in range(n, 0, -1):          # start from largest subset size for a quick win
        for subset in combinations(activities, size):
            ordered = sorted(subset, key=lambda a: a[1])
            ok = True
            for i in range(1, len(ordered)):
                if ordered[i][0] < ordered[i - 1][1]:  # overlap detected
                    ok = False
                    break
            if ok:
                return size            # first (largest) compatible subset found
    return best


def shortest_duration_first(activities: List[Tuple[int, int]]) -> List[Tuple[int, int]]:
    """A plausible-sounding but INCORRECT greedy strategy, kept here as a counterexample."""
    ordered = sorted(activities, key=lambda a: a[1] - a[0])  # sort by duration, not finish time
    selected = []
    last_finish = -1
    for start, finish in ordered:
        if start >= last_finish:
            selected.append((start, finish))
            last_finish = max(last_finish, finish)
    return selected


# A small instance where "shortest duration first" is provably suboptimal: the shortest
# activity (2, 4) blocks both (0, 3) and (3, 6), which together tile the timeline better.
tricky = [(0, 3), (2, 4), (3, 6)]

correct = activity_selection(tricky)
wrong_strategy = shortest_duration_first(tricky)
optimal_size = brute_force_max_compatible(tricky)

print("earliest-finish-first  ->", correct, " size =", len(correct))
print("shortest-duration-first ->", wrong_strategy, " size =", len(wrong_strategy))
print("brute-force optimum size:", optimal_size)

assert len(correct) == optimal_size, "earliest-finish-first should match brute force"
# The short activity (2, 4) looks tempting because it has the shortest duration, but taking
# it blocks both (0, 3) and (3, 6), which together tile the timeline with no gap or overlap.
assert len(wrong_strategy) < optimal_size, "this counterexample should defeat shortest-duration-first"
print("Counterexample confirmed: shortest-duration-first is not optimal, earliest-finish-first is.")


## 3. Fractional knapsack vs. 0/1 knapsack

**Fractional knapsack:** given items with weights and values, and a knapsack of capacity
`W`, choose *fractions* of items (you may take 40% of an item) to maximize total value
without exceeding `W`. This has the greedy-choice property: sort items by value-per-unit-weight
(value density) descending, and greedily fill the knapsack, taking a fraction of the last
item that fits. Because you can always take a bit more of the highest-density item
remaining, no exchange can ever improve on this — swapping any weight from a lower-density
item to a higher-density one strictly increases value.

**0/1 knapsack:** the same setup, but each item must be taken whole or not at all. This
small change *breaks* the greedy-choice property: the highest-density item might not fit
the remaining capacity at all, and taking it can force you to leave a much larger amount of
capacity under-used compared to a different combination. This is exactly why 0/1 knapsack
needs dynamic programming (Week 08's table over `(item, remaining capacity)` states) rather
than a single sorted greedy pass — a locally best choice can be globally wrong.


In [ ]:
def fractional_knapsack(weights: List[float], values: List[float], capacity: float) -> float:
    """Greedy solution: sort by value density, take as much of each item as fits."""
    n = len(weights)
    # Sort item indices by value-per-weight, descending -- the greedy-choice property holds here.
    order = sorted(range(n), key=lambda i: values[i] / weights[i], reverse=True)
    total_value = 0.0
    remaining = capacity
    for i in order:
        if weights[i] <= remaining:
            total_value += values[i]      # take the whole item
            remaining -= weights[i]
        else:
            fraction = remaining / weights[i]
            total_value += fraction * values[i]   # take only the fraction that fits
            remaining = 0.0
            break                          # knapsack is full, nothing more can be added
    return total_value


def zero_one_knapsack_dp(weights: List[int], values: List[int], capacity: int) -> int:
    """Reference DP solution from Week 08, used here only for the greedy-vs-DP comparison."""
    n = len(weights)
    table = [[0] * (capacity + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for c in range(capacity + 1):
            table[i][c] = table[i - 1][c]                    # option: skip item i-1
            if weights[i - 1] <= c:                           # option: take item i-1, if it fits
                table[i][c] = max(table[i][c], table[i - 1][c - weights[i - 1]] + values[i - 1])
    return table[n][capacity]


def zero_one_greedy_by_density(weights: List[int], values: List[int], capacity: int) -> int:
    """The INCORRECT greedy strategy applied to 0/1 knapsack -- kept as a counterexample."""
    n = len(weights)
    order = sorted(range(n), key=lambda i: values[i] / weights[i], reverse=True)
    remaining = capacity
    total = 0
    for i in order:
        if weights[i] <= remaining:      # can only take the item whole, or not at all
            total += values[i]
            remaining -= weights[i]
    return total


weights = [10, 20, 30]
values = [60, 100, 120]
W = 50

frac_value = fractional_knapsack(weights, values, W)
dp_value = zero_one_knapsack_dp(weights, values, W)
greedy_01_value = zero_one_greedy_by_density(weights, values, W)

print(f"fractional knapsack (greedy, provably optimal): {frac_value}")
print(f"0/1 knapsack (dynamic programming, optimal):     {dp_value}")
print(f"0/1 knapsack (greedy by density, NOT optimal):    {greedy_01_value}")

# Greedy-by-density picks items 2 and 1 (values 100 + 60 = 160) then cannot fit item 0's
# remaining weight, missing the true optimum of 220 (items 1 and 2 exactly fill 50).
assert dp_value == 220
assert greedy_01_value < dp_value, "greedy-by-density should be suboptimal for 0/1 knapsack"
print("Confirmed: value density is the right sort key for fractional knapsack, but greedy on")
print("the same key is NOT safe for 0/1 knapsack -- 0/1 genuinely needs the DP table.")


## 4. Coin change: greedy works only for canonical systems

**Problem:** given a set of coin denominations and a target amount, use the fewest coins
that sum exactly to the target (unlimited supply of each denomination).

The greedy strategy — repeatedly take the largest denomination that does not exceed the
remaining amount — is fast and simple, and it happens to be optimal for many real-world
currencies, including US cents (1, 5, 10, 25) and Indian rupees (1, 2, 5, 10, 20). Coin
systems for which greedy is always optimal are called **canonical**. But greedy is *not*
correct in general: a system like `{1, 3, 4}` breaks it, because it is not canonical.
Week 08's dynamic-programming solution to coin change is correct for *any* coin system,
canonical or not, because it explores every combination via the table rather than
committing to the largest coin first. This is a clean, concrete illustration of the
greedy-vs-DP decision framework from Section 1: use greedy only when you can prove (or, at
minimum, verify against DP) that the greedy-choice property holds for your specific input.


In [ ]:
def greedy_coin_change(coins: List[int], target: int) -> List[int]:
    """Greedy coin change: repeatedly take the largest coin that fits. NOT always optimal."""
    coins = sorted(coins, reverse=True)
    remaining = target
    used = []
    for c in coins:
        while c <= remaining:            # take this denomination as many times as it fits
            used.append(c)
            remaining -= c
    return used if remaining == 0 else []   # empty list signals "greedy could not make exact change"


def dp_coin_change_min_count(coins: List[int], target: int) -> int:
    """Minimum-coin DP from Week 08 -- correct for any coin system, canonical or not."""
    INF = float("inf")
    fewest = [0] + [INF] * target
    for amount in range(1, target + 1):
        for c in coins:
            if c <= amount and fewest[amount - c] + 1 < fewest[amount]:
                fewest[amount] = fewest[amount - c] + 1     # try using one more coin c
    return fewest[target]


# A canonical system: greedy and DP agree.
us_like = [1, 5, 10, 25]
target = 41
g = greedy_coin_change(us_like, target)
d = dp_coin_change_min_count(us_like, target)
print(f"US-like coins, target {target}: greedy uses {len(g)} coins {g}, DP optimum is {d}")
assert len(g) == d

# A non-canonical system: greedy is provably wrong.
non_canonical = [1, 3, 4]
target = 6
g2 = greedy_coin_change(non_canonical, target)
d2 = dp_coin_change_min_count(non_canonical, target)
print(f"Non-canonical coins {{1,3,4}}, target {target}: greedy uses {len(g2)} coins {g2}, DP optimum is {d2}")

# Greedy takes 4 then 1+1 (three coins: 4, 1, 1); the true optimum is 3+3 (two coins).
assert len(g2) > d2, "greedy should be strictly worse than DP on this non-canonical system"
print("Confirmed: greedy fails on a non-canonical coin system -- DP is required in general.")


## 5. Huffman coding

**Problem:** given the frequency of each symbol in a message, build a binary **prefix
code** (no code is a prefix of another, so the encoded bit-stream can be decoded
unambiguously) that minimizes the total number of bits needed to encode the message.

The greedy idea: repeatedly take the two *least frequent* remaining symbols (or partial
trees) and merge them into a new tree node whose frequency is their sum, using a min-heap
to always find the two smallest efficiently (the same heap structure from Week 08). Rarer
symbols end up deeper in the tree and get longer codes; common symbols end up shallow and
get short codes — this is exactly what a good prefix code should do. The exchange argument
here shows that the two least-frequent symbols can always be made siblings at maximum
depth in *some* optimal tree, which justifies merging them first.

A common pitfall: forgetting that Huffman coding is only optimal for *this* message's
frequency table. It is not a universal compressor, and ties in frequency can produce
different (but equally optimal, in total bit count) trees depending on tie-breaking order.


In [ ]:
import heapq
from typing import Dict


def build_huffman_codes(freqs: Dict[str, int]) -> Dict[str, str]:
    """Build a Huffman code table from symbol frequencies.

    Returns a dict mapping each symbol to its binary code string.
    """
    # Each heap entry is [frequency, [ [symbol, code_so_far], ... ] ] -- a list of symbols
    # sharing a subtree, each tracking the code bits accumulated so far.
    heap = [[freq, [[symbol, ""]]] for symbol, freq in freqs.items()]
    heapq.heapify(heap)

    if len(heap) == 1:
        # Degenerate case: only one symbol -- give it a 1-bit code by convention.
        only = heap[0][1]
        return {only[0][0]: "0"}

    while len(heap) > 1:
        lo = heapq.heappop(heap)   # two least-frequent (sub)trees
        hi = heapq.heappop(heap)
        for pair in lo[1]:
            pair[1] = "0" + pair[1]     # everything in the lighter subtree gets a leading 0
        for pair in hi[1]:
            pair[1] = "1" + pair[1]     # everything in the heavier subtree gets a leading 1
        merged = [lo[0] + hi[0], lo[1] + hi[1]]   # combined frequency, combined symbol list
        heapq.heappush(heap, merged)

    return dict(heap[0][1])


def huffman_encode(message: str, codes: Dict[str, str]) -> str:
    return "".join(codes[ch] for ch in message)


def huffman_decode(bits: str, codes: Dict[str, str]) -> str:
    """Decode by walking the bit string, matching against the code table one prefix at a time."""
    reverse = {code: symbol for symbol, code in codes.items()}
    decoded = []
    buffer = ""
    for bit in bits:
        buffer += bit
        if buffer in reverse:            # prefix-free property guarantees no ambiguity here
            decoded.append(reverse[buffer])
            buffer = ""
    return "".join(decoded)


message = "abracadabra"
freqs = {ch: message.count(ch) for ch in set(message)}
codes = build_huffman_codes(freqs)
print("frequencies:", freqs)
print("codes:      ", codes)

encoded = huffman_encode(message, codes)
decoded = huffman_decode(encoded, codes)
print(f"original:  {message!r} ({len(message) * 8} bits as fixed-width ASCII)")
print(f"encoded:   {encoded} ({len(encoded)} bits)")
print(f"decoded:   {decoded!r}")

assert decoded == message
assert len(encoded) < len(message) * 8
print("Confirmed: Huffman coding round-trips correctly and beats fixed-width encoding here.")


## 6. A live timing comparison: greedy vs. exhaustive search

The whole appeal of a correct greedy algorithm is speed: a single sorted pass, no
backtracking. The cell below times `activity_selection` (greedy, roughly O(n log n) from
the sort) against `brute_force_max_compatible` (exponential) on the same growing input, to
make that speed difference concrete rather than just asserted.


In [ ]:
import timeit
import random

random.seed(0)

def make_activities(n: int) -> List[Tuple[int, int]]:
    acts = []
    for _ in range(n):
        start = random.randint(0, 50)
        finish = start + random.randint(1, 10)
        acts.append((start, finish))
    return acts


for n in [4, 8, 12]:
    acts = make_activities(n)
    greedy_time = timeit.timeit(lambda: activity_selection(acts), number=200)
    brute_time = timeit.timeit(lambda: brute_force_max_compatible(acts), number=5 if n <= 8 else 1)
    print(f"n={n:2d}: greedy avg {greedy_time / 200 * 1e6:8.2f} microseconds  |  "
          f"brute force avg {brute_time / (200 if n <= 8 else 1) * 1e3:8.3f} ms (fewer reps at n=12: too slow otherwise)")


(Timings vary by machine, but the qualitative trend is stable: brute force's cost grows
sharply with `n` because it enumerates subsets, while greedy stays fast because it only
sorts and scans once. This is the practical payoff of proving the greedy-choice property
instead of resorting to exhaustive search.)


## Exercises

### Exercise 1 — Weighted interval scheduling detector

Plain activity selection assumes every activity is worth the same. Write
`can_greedy_maximize_count(activities)` that returns `True` if maximizing the *count* of
selected activities (ignoring any notion of value) is what greedy activity selection
solves, which is always the case for the classic problem — but as a sanity-building
exercise, implement a function that reports, for a given instance, how many activities the
earliest-finish-time greedy selects, **and** verifies that number against brute force,
raising `ValueError` if they disagree (which should never happen, but the check makes the
guarantee explicit).

Example:
```
verify_activity_selection([(1, 4), (3, 5), (0, 6), (5, 7), (8, 9)])
# -> 3   (greedy count, confirmed equal to brute-force optimum)
```


In [ ]:
def verify_activity_selection(activities: List[Tuple[int, int]]) -> int:
    """Run greedy activity selection, verify its count against brute force, return the count.

    Args:
        activities: list of (start, finish) pairs.

    Returns:
        The number of activities selected by the greedy algorithm.

    Raises:
        ValueError: if the greedy count does not match the brute-force optimum
            (should never happen for a correctly implemented greedy algorithm --
            this is a correctness guard, not expected behavior).
    """
    # TODO: implement this
    raise NotImplementedError


### Self-Check — Exercise 1

In [ ]:
result = verify_activity_selection([(1, 4), (3, 5), (0, 6), (5, 7), (8, 9)])
assert result == 3

result2 = verify_activity_selection([(0, 10), (0, 2), (2, 4), (4, 6), (6, 8), (8, 10)])
assert result2 == 5

print("✅ Exercise 1 passed")


### Exercise 2 — Fractional knapsack with item labels

Extend fractional knapsack to also report *which* items (and what fraction of each) were
taken, not just the total value. Write `fractional_knapsack_plan(items, capacity)` where
`items` is a list of `(name, weight, value)` tuples, returning a tuple
`(total_value, plan)` where `plan` is a list of `(name, fraction_taken)` for every item
with `fraction_taken > 0`, in the order the items were taken (highest value density first).

Example:
```
fractional_knapsack_plan([("gold", 10, 60), ("silver", 20, 100), ("gems", 30, 120)], 50)
# -> (240.0, [("gold", 1.0), ("silver", 1.0), ("gems", 0.6666666666666666)])
```


In [ ]:
def fractional_knapsack_plan(items: List[Tuple[str, float, float]], capacity: float):
    """Solve fractional knapsack and report the fraction of each item taken.

    Args:
        items: list of (name, weight, value) tuples.
        capacity: knapsack capacity.

    Returns:
        (total_value, plan) where plan is a list of (name, fraction_taken) tuples,
        in the order items were taken, omitting any item with fraction 0.
    """
    # TODO: implement this
    raise NotImplementedError


### Self-Check — Exercise 2

In [ ]:
total, plan = fractional_knapsack_plan(
    [("gold", 10, 60), ("silver", 20, 100), ("gems", 30, 120)], 50
)
assert abs(total - 240.0) == 0 or abs(total - 240.0) < 1e-9
assert plan[0] == ("gold", 1.0)
assert plan[1] == ("silver", 1.0)
assert abs(plan[2][1] - 2 / 3) < 1e-9

print("✅ Exercise 2 passed")


### Exercise 3 — Coin system canonicity checker

Write `is_canonical(coins, max_amount)` that checks, for every target amount from 1 up to
`max_amount`, whether the greedy coin-change algorithm uses the same number of coins as the
optimal (DP) solution. Return `True` if greedy matches DP for every amount tested, `False`
otherwise. This is a *practical* (not a proof-based) canonicity check: it can only tell you
a system is non-canonical by finding a counterexample within the tested range, it cannot
prove canonicity for all amounts beyond `max_amount`.

Example:
```
is_canonical([1, 5, 10, 25], 100)   # -> True
is_canonical([1, 3, 4], 10)         # -> False   (fails at amount 6: greedy uses 3, DP uses 2)
```


In [ ]:
def is_canonical(coins: List[int], max_amount: int) -> bool:
    """Check whether greedy coin change matches the DP optimum for every amount in range.

    Args:
        coins: list of coin denominations.
        max_amount: check every target amount from 1 to max_amount inclusive.

    Returns:
        True if greedy's coin count equals the DP optimum for every tested amount,
        False if a counterexample is found.
    """
    # TODO: implement this
    raise NotImplementedError


### Self-Check — Exercise 3

In [ ]:
assert is_canonical([1, 5, 10, 25], 100) is True
assert is_canonical([1, 3, 4], 10) is False

print("✅ Exercise 3 passed")


### Exercise 4 — Huffman average code length

Write `average_code_length(freqs)` that builds a Huffman code from a frequency table and
returns the **weighted average number of bits per symbol** in the resulting code (total
encoded bits for one occurrence of each symbol at its given frequency, divided by the total
number of symbol occurrences). This number is what you would compare against `log2(alphabet
size)` (the fixed-width baseline) to quantify the compression benefit.

Example (approximate, ties may shift the exact tree but not this number):
```
average_code_length({"a": 5, "b": 2, "c": 1, "d": 1})
# -> 1.6666666666666667
```


In [ ]:
def average_code_length(freqs: Dict[str, int]) -> float:
    """Build a Huffman code and compute the frequency-weighted average code length in bits.

    Args:
        freqs: mapping from symbol to occurrence count.

    Returns:
        The weighted average number of bits per symbol under the Huffman code.
    """
    # TODO: implement this
    raise NotImplementedError


### Self-Check — Exercise 4

In [ ]:
avg = average_code_length({"a": 5, "b": 2, "c": 1, "d": 1})
assert abs(avg - 5 / 3) < 1e-9   # 5*1 + 2*2 + 1*3 + 1*3 = 15 bits total, 9 symbol occurrences

single = average_code_length({"z": 7})
assert single == 1.0   # degenerate one-symbol case: 1 bit per symbol by convention

print("✅ Exercise 4 passed")


### Exercise 5 (harder) — Greedy vs. DP judgment calls

Write `recommend_strategy(problem_name)` that takes one of the exact strings
`"activity_selection"`, `"fractional_knapsack"`, `"0/1_knapsack"`, `"coin_change_canonical"`,
`"coin_change_arbitrary"`, or `"huffman_coding"`, and returns either `"greedy"` or `"dp"`,
matching the judgment calls made throughout this notebook. Raise `ValueError` for any other
input string. This exercise has no computation — it checks that you can articulate *why*
each problem falls on one side of the greedy/DP line, which is the real skill this week is
building.


In [ ]:
def recommend_strategy(problem_name: str) -> str:
    """Return 'greedy' or 'dp' for a named classic optimization problem.

    Args:
        problem_name: one of "activity_selection", "fractional_knapsack",
            "0/1_knapsack", "coin_change_canonical", "coin_change_arbitrary",
            "huffman_coding".

    Returns:
        "greedy" if the problem is correctly and efficiently solved by a greedy
        algorithm, "dp" if it requires dynamic programming in general.

    Raises:
        ValueError: if problem_name is not one of the recognized strings.
    """
    # TODO: implement this
    raise NotImplementedError


### Self-Check — Exercise 5

In [ ]:
assert recommend_strategy("activity_selection") == "greedy"
assert recommend_strategy("fractional_knapsack") == "greedy"
assert recommend_strategy("0/1_knapsack") == "dp"
assert recommend_strategy("coin_change_canonical") == "greedy"
assert recommend_strategy("coin_change_arbitrary") == "dp"
assert recommend_strategy("huffman_coding") == "greedy"

try:
    recommend_strategy("bin_packing")
    raise AssertionError("expected ValueError for an unrecognized problem name")
except ValueError:
    print("correctly rejected an unrecognized problem name")

print("✅ Exercise 5 passed")


## Quiz

**1. Why does sorting activities by finish time (rather than by start time or duration)
make the activity-selection greedy algorithm correct?**

<details><summary>Show answer</summary>
Finishing earliest leaves the maximum possible remaining time for future activities. The
exchange argument shows that any optimal solution can be transformed, without loss, into
one that starts with the earliest-finishing activity, so choosing it first never costs
anything and is always at least as good as any alternative first choice.
</details>

**2. Why does the greedy value-density strategy work for fractional knapsack but fail for
0/1 knapsack?**

<details><summary>Show answer</summary>
In fractional knapsack you can always use up remaining capacity exactly by taking a
fraction of the next-best item, so higher density always wins without waste. In 0/1
knapsack an item must be taken whole; the highest-density item might not fit the remaining
capacity, forcing you to either waste capacity or skip it, and a lower-density combination
can pack the knapsack better. This waste is exactly what breaks the greedy-choice property,
which is why 0/1 knapsack needs the DP table over items and capacity instead.
</details>

**3. A new coin system is `{1, 4, 6}`. Is it canonical? (Try the target amount 8 by hand
before checking the answer, or verify with `is_canonical` from Exercise 3.)**

<details><summary>Show answer</summary>
No. Greedy picks 6, then 1, then 1 (three coins for 8), but 4 + 4 makes 8 in two coins.
Verifying with `is_canonical([1, 4, 6], 20)` returns `False`, with 8 as one counterexample
among possibly others.
</details>

**4. In Huffman coding, why do the two least-frequent symbols always end up as siblings at
maximum depth in an optimal code tree?**

<details><summary>Show answer</summary>
Because giving the two least-frequent symbols the longest codes (deepest in the tree) costs
the least in total weighted bit length — any other pair swapped into those deepest
positions would have to be at least as frequent, increasing the total. The exchange
argument used to justify Huffman's greedy merge step formalizes exactly this: if the two
least-frequent symbols were not siblings at the deepest level in some claimed-optimal tree,
swapping them there would not increase (and could only decrease) the total encoded length,
contradicting optimality unless they were already positioned that way.
</details>


## Solutions (try the exercises yourself first!)

### Solution 1

In [ ]:
def verify_activity_selection(activities: List[Tuple[int, int]]) -> int:
    greedy_result = activity_selection(activities)
    greedy_count = len(greedy_result)
    brute_count = brute_force_max_compatible(activities)
    if greedy_count != brute_count:
        raise ValueError(
            f"greedy count {greedy_count} does not match brute-force optimum {brute_count}"
        )
    return greedy_count


result = verify_activity_selection([(1, 4), (3, 5), (0, 6), (5, 7), (8, 9)])
assert result == 3
print("✅ Solution 1 verified")


### Solution 2

In [ ]:
def fractional_knapsack_plan(items: List[Tuple[str, float, float]], capacity: float):
    order = sorted(items, key=lambda item: item[2] / item[1], reverse=True)
    remaining = capacity
    total_value = 0.0
    plan = []
    for name, weight, value in order:
        if remaining <= 0:
            break
        if weight <= remaining:
            total_value += value           # take the whole item
            remaining -= weight
            plan.append((name, 1.0))
        else:
            fraction = remaining / weight
            total_value += fraction * value
            plan.append((name, fraction))
            remaining = 0.0
    return total_value, plan


total, plan = fractional_knapsack_plan(
    [("gold", 10, 60), ("silver", 20, 100), ("gems", 30, 120)], 50
)
print(total, plan)
assert plan[0] == ("gold", 1.0)
print("✅ Solution 2 verified")


### Solution 3

In [ ]:
def is_canonical(coins: List[int], max_amount: int) -> bool:
    for amount in range(1, max_amount + 1):
        greedy_result = greedy_coin_change(coins, amount)
        if not greedy_result:
            continue     # greedy could not make this amount at all; skip, not a canonicity failure here
        dp_optimum = dp_coin_change_min_count(coins, amount)
        if len(greedy_result) != dp_optimum:
            return False
    return True


assert is_canonical([1, 5, 10, 25], 100) is True
assert is_canonical([1, 3, 4], 10) is False
print("✅ Solution 3 verified")


### Solution 4

In [ ]:
def average_code_length(freqs: Dict[str, int]) -> float:
    codes = build_huffman_codes(freqs)
    total_bits = sum(freqs[symbol] * len(code) for symbol, code in codes.items())
    total_symbols = sum(freqs.values())
    return total_bits / total_symbols


avg = average_code_length({"a": 5, "b": 2, "c": 1, "d": 1})
print(avg)
assert abs(avg - 5 / 3) < 1e-9
print("✅ Solution 4 verified")


### Solution 5

In [ ]:
def recommend_strategy(problem_name: str) -> str:
    greedy_problems = {
        "activity_selection",
        "fractional_knapsack",
        "coin_change_canonical",
        "huffman_coding",
    }
    dp_problems = {
        "0/1_knapsack",
        "coin_change_arbitrary",
    }
    if problem_name in greedy_problems:
        return "greedy"
    if problem_name in dp_problems:
        return "dp"
    raise ValueError(f"unrecognized problem name: {problem_name!r}")


assert recommend_strategy("0/1_knapsack") == "dp"
assert recommend_strategy("huffman_coding") == "greedy"
print("✅ Solution 5 verified")
